# 10 — SOTA baselines under a common setting (R2.2 / R3)

Runs each published system's **released code** on our question sets and scores its predictions with **our** execution + schema-linking metrics on **our** DBs, so differences reflect the method, not the harness. Converters match the real output formats of CHESS (`ShayanTalaei/CHESS`) and RSL-SQL (`Laqcce-cao/RSL-SQL`); the others use a one-line-per-question JSONL.

**Note:** this notebook loads notebook 02's scoring functions *without* re-running generation (it executes 02 only up to the generation loop), so your `generation_results.csv` is never overwritten.

## Load config + 02's scoring helpers (no generation re-run)

In [1]:
# Load config (00) and notebook 02's SCORING FUNCTIONS only -- WITHOUT re-running
# the generation loop (which would overwrite generation_results.csv). We read 02's
# source and execute everything UP TO the generation loop.
import re, nbformat
%run 00_setup_and_config.ipynb

_nb02 = nbformat.read("02_run_generation.ipynb", as_version=4)
_src = "\n".join(c.source for c in _nb02.cells if c.cell_type == "code")
_marker = "global_bar = tqdm(total=PLAN_TOTAL"   # cut before the OVERALL progress bar
assert _marker in _src, "generation-loop marker not found in 02; check the notebook."
_src = _src[:_src.index(_marker)]                      # drop the loop + CSV save
_src = "\n".join("" if re.match(r"\s*[%!]", ln) else ln for ln in _src.split("\n"))
exec(compile(_src, "02_helpers", "exec"), globals())   # defines clean_sql, validate_sql,
                                                       # run_sql, execution_match, load_questions,
                                                       # link_scores_strict, table_scores, ...
print("Loaded 02 helpers:",
      [f for f in ["clean_sql","validate_sql","execution_match","load_questions",
                   "link_scores_strict","table_scores"] if f in globals()])

Core libraries imported.
Project directories ready.
Environment loaded. Model tiers:
  WEAK   -> gpt-4o-mini
  MID    -> gpt-5.4-mini
  STRONG -> gpt-5.4
  REASON -> gpt-6-luna
Conditions: ['C1', 'C2', 'C3', 'C4', 'C5']
Datasets   : ['D1', 'D2', 'D2_fin', 'D3']
Grayscale plotting style configured.
Condition styles: {'C1': '0.15', 'C2': '0.3', 'C3': '0.45', 'C4': '0.6', 'C5': '0.75'}
Live check skipped (set RUN_LIVE_CHECK=True to test credentials).
Setup complete. Config snapshot saved.
Notebook 02 dependencies ready.
Mode: FULL | cells=105 | limit=None
Planned: 105 cells, 65742 question-generations total.
Loaded 02 helpers: ['clean_sql', 'validate_sql', 'execution_match', 'load_questions', 'link_scores_strict', 'table_scores']


## Scoring harness (our metrics)

In [2]:
import json, numpy as np, pandas as pd
from pathlib import Path

SOTA_DIR = RESULTS_DIR/"sota"     # put each system's predictions here

def score_predictions(system, dataset_key, preds):
    """preds: list of {qid, db_id, pred_sql, selected_tables?, selected_columns?, context_tokens?}.
    Scored with OUR metrics on OUR DBs so differences reflect the method, not the harness."""
    questions = load_questions(dataset_key); rows = []
    for r in preds:
        q = questions.get(r["qid"])
        if q is None: continue
        sql = clean_sql(r.get("pred_sql", "")); ok, verr = validate_sql(sql)
        if ok:
            match, perr, _ = execution_match(dataset_key, q["db_id"], sql, q["gold_sql"])
        else:
            match, perr = False, verr
        t_rec, t_prec = table_scores(r.get("selected_tables", []), q.get("gold_tables", []))
        cs_rec, cs_prec = link_scores_strict(r.get("selected_columns", []), q.get("gold_columns", []))
        rows.append({"dataset": dataset_key, "condition": system, "model_tier": "SOTA",
                     "qid": r["qid"], "db_id": q["db_id"],
                     "exec_match": (np.nan if match is None else int(bool(match))),
                     "valid_sql": int(ok), "table_recall": t_rec, "table_precision": t_prec,
                     "col_recall_strict": cs_rec, "col_precision_strict": cs_prec,
                     "context_tokens_est": r.get("context_tokens", np.nan),
                     "pred_sql": sql, "pred_error": perr})
    return pd.DataFrame(rows)

## Converters matched to each system's released output format

In [3]:
# ---- Converters for the released code of each system -> our canonical preds -----
def from_CHESS(predictions_json):
    """CHESS writes BIRD-style {qid: "<sql>\t----- bird -----\t<db_id>"}
    (src/runner/run_manager.py: update_final_predictions)."""
    d = json.loads(Path(predictions_json).read_text()); out = []
    for qid, val in d.items():
        out.append({"qid": str(qid), "pred_sql": val.split("\t----- bird -----\t")[0].strip()})
    return out

def from_RSL_SQL(result_dir):
    """RSL-SQL main_result/<split>/: ppl_dev.json [{id, db, ...}] aligned with
    LLM.json [{tables, columns}] (src/bid_schema_linking.py)."""
    rd = Path(result_dir)
    ppl = json.loads((rd/"ppl_dev.json").read_text())
    link = json.loads((rd/"LLM.json").read_text()) if (rd/"LLM.json").exists() else [{}]*len(ppl)
    return [{"qid": str(i["id"]), "db_id": i.get("db"),
             "pred_sql": i.get("pred_sql") or i.get("sql") or "",
             "selected_tables": lk.get("tables", []), "selected_columns": lk.get("columns", [])}
            for i, lk in zip(ppl, link)]

def from_generic_jsonl(path):
    """CM-SQL / SchemaGraphSQL / AutoLink / Graph-Link: one JSON per line with
    {qid, db_id, pred_sql, selected_tables?, selected_columns?, context_tokens?}."""
    return [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]

## Produce the head-to-head comparison

In [4]:
# ---- Build the head-to-head table (uncomment once each predictions file exists) ----
JOBS = [
    # ("CHESS",         "D2", from_CHESS(str(SOTA_DIR/"CHESS/bird_predictions.json"))),
    # ("RSL_SQL",       "D2", from_RSL_SQL(str(SOTA_DIR/"RSL-SQL/main_result/bird"))),
    # ("CM_SQL",        "D2", from_generic_jsonl(str(SOTA_DIR/"CM_SQL/D2.jsonl"))),
    # ("SchemaGraphSQL","D2", from_generic_jsonl(str(SOTA_DIR/"SchemaGraphSQL/D2.jsonl"))),
    # ("AutoLink",      "D2", from_generic_jsonl(str(SOTA_DIR/"AutoLink/D2.jsonl"))),
    # ("GraphLink",     "D2", from_generic_jsonl(str(SOTA_DIR/"GraphLink/D2.jsonl"))),
]
frames = [score_predictions(sys, ds, preds) for sys, ds, preds in JOBS]
if frames:
    sota = pd.concat(frames, ignore_index=True)
    save_table(sota, "NEW_sota_predictions_scored")
    cmp = sota.groupby(["dataset","condition"]).agg(
        EX=("exec_match","mean"), col_recall=("col_recall_strict","mean"),
        tbl_recall=("table_recall","mean")).round(3).reset_index()
    display(cmp)
    print("Compare against our C1 (full) and C4 (hybrid) from generation_results.csv.")
else:
    print("Uncomment JOBS once each system's predictions file exists under results/sota/<system>/.")

Uncomment JOBS once each system's predictions file exists under results/sota/<system>/.
